# Chapter 5.6: ModelRunner & TpModelWorker

## Learning Objectives

By the end of this notebook, you will:

1. **Understand TpModelWorker** and its role in tensor parallel execution
2. **Master ModelRunner** and how forward passes are executed
3. **Learn input preparation**: token IDs, positions, attention metadata
4. **Differentiate extend (prefill) vs decode** forward pass modes
5. **Understand CUDA graph capture** for decode acceleration
6. **Trace model output processing** from logits to sampled tokens

---

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hideak1/vllm_study/blob/main/notebooks/part5/chapter_5.6_model_runner.ipynb)
[![Download Notebook](https://img.shields.io/badge/Download-Notebook-blue)](https://raw.githubusercontent.com/hideak1/vllm_study/main/notebooks/part5/chapter_5.6_model_runner.ipynb)

**How to do the exercises:**
1. **Google Colab (Recommended)**: Click the "Open in Colab" badge above — you get your own copy with free GPU.
2. **Local Jupyter**: Clone the repo, run `./start.sh`, then open notebooks at `http://localhost:8888`.
3. **Exercises**: Look for cells marked with 🏋️ **Exercise** or **Assignment**. Fill in the `TODO` sections and run the cell to check your work.

> **Tip**: In Colab, the notebook is automatically copied to your Drive — your changes are saved there.

## 1. TpModelWorker Overview

The `TpModelWorker` (Tensor Parallel Model Worker) is responsible for:
- Loading the model onto GPU(s)
- Managing tensor parallel communication
- Providing the `ModelRunner` for forward pass execution
- Allocating GPU memory for KV-cache

```python
# sglang/srt/managers/tp_worker.py (simplified)

class TpModelWorker:
    def __init__(self, server_args, gpu_id, tp_rank, dp_rank=0):
        # Set GPU device
        self.gpu_id = gpu_id
        self.tp_rank = tp_rank
        torch.cuda.set_device(gpu_id)
        
        # Load model configuration
        self.model_config = ModelConfig(
            server_args.model_path,
            trust_remote_code=server_args.trust_remote_code
        )
        
        # Create ModelRunner (handles forward pass)
        self.model_runner = ModelRunner(
            model_config=self.model_config,
            server_args=server_args,
            tp_rank=tp_rank,
            tp_size=server_args.tp_size,
        )
        
        # Initialize memory pools
        self._init_memory_pool(server_args)
    
    def _init_memory_pool(self, server_args):
        """Allocate GPU memory for KV-cache."""
        # Calculate available memory
        total_gpu_mem = torch.cuda.get_device_properties(self.gpu_id).total_memory
        model_mem = self.model_runner.get_model_memory()
        available = total_gpu_mem * server_args.mem_fraction_static - model_mem
        
        # Calculate max tokens
        kv_bytes_per_token = self.model_config.get_kv_cache_bytes_per_token()
        self.max_total_num_tokens = int(available / kv_bytes_per_token)
        
        # Create pools
        self.token_to_kv_pool = TokenToKVPool(
            self.max_total_num_tokens, ...)
        self.req_to_token_pool = ReqToTokenPool(
            self.max_total_num_tokens, ...)
    
    def forward_batch(self, batch):
        """Execute forward pass for a batch."""
        return self.model_runner.forward(batch)
```

In [ ]:
# Simulate TpModelWorker initialization

from dataclasses import dataclass
from typing import List, Optional, Dict
import math

@dataclass
class GPUInfo:
    """Simulated GPU information."""
    gpu_id: int
    name: str
    total_memory_gb: float
    
    @property
    def total_memory_bytes(self):
        return int(self.total_memory_gb * 1024**3)

class TpModelWorkerSimulator:
    """Simulates TpModelWorker initialization."""
    
    def __init__(self, model_name, gpu_info, tp_rank, tp_size, mem_fraction=0.88):
        self.model_name = model_name
        self.gpu_info = gpu_info
        self.tp_rank = tp_rank
        self.tp_size = tp_size
        self.mem_fraction = mem_fraction
        
        # Model configs (simplified)
        model_configs = {
            "Llama-3.1-8B": {
                "params_b": 8.0, "layers": 32, "heads": 32, 
                "kv_heads": 8, "head_dim": 128, "hidden": 4096
            },
            "Llama-3.1-70B": {
                "params_b": 70.0, "layers": 80, "heads": 64, 
                "kv_heads": 8, "head_dim": 128, "hidden": 8192
            },
        }
        self.config = model_configs.get(model_name, model_configs["Llama-3.1-8B"])
    
    def calculate_memory_allocation(self):
        """Calculate memory allocation for this TP rank."""
        # Model weight memory (per GPU with TP)
        model_size_bytes = self.config["params_b"] * 1e9 * 2  # FP16 = 2 bytes
        model_per_gpu = model_size_bytes / self.tp_size
        
        # KV-cache per token (per GPU)
        kv_heads_per_gpu = self.config["kv_heads"] // self.tp_size
        kv_per_token = 2 * self.config["layers"] * kv_heads_per_gpu * self.config["head_dim"] * 2
        
        # Available for KV-cache
        total = self.gpu_info.total_memory_bytes
        static_memory = total * self.mem_fraction
        available_for_kv = static_memory - model_per_gpu
        
        # Max tokens
        max_tokens = int(available_for_kv / kv_per_token)
        
        return {
            "gpu": f"GPU {self.gpu_info.gpu_id} ({self.gpu_info.name})",
            "tp_rank": self.tp_rank,
            "total_gpu_gb": self.gpu_info.total_memory_gb,
            "model_per_gpu_gb": model_per_gpu / 1024**3,
            "kv_per_token_bytes": kv_per_token,
            "available_for_kv_gb": available_for_kv / 1024**3,
            "max_tokens": max_tokens,
            "kv_heads_per_gpu": kv_heads_per_gpu,
        }

# Simulate for different configs
configs = [
    ("Llama-3.1-8B", 1, GPUInfo(0, "A100-80GB", 80)),
    ("Llama-3.1-70B", 4, GPUInfo(0, "A100-80GB", 80)),
]

print("TpModelWorker Memory Allocation")
print("=" * 75)

for model_name, tp_size, gpu in configs:
    print(f"\n--- {model_name} (TP={tp_size}) ---")
    for tp_rank in range(min(tp_size, 2)):  # Show first 2 ranks
        worker = TpModelWorkerSimulator(model_name, gpu, tp_rank, tp_size)
        alloc = worker.calculate_memory_allocation()
        print(f"  TP Rank {alloc['tp_rank']}: {alloc['gpu']}")
        print(f"    Model weights: {alloc['model_per_gpu_gb']:.1f} GB")
        print(f"    KV-cache available: {alloc['available_for_kv_gb']:.1f} GB")
        print(f"    KV per token: {alloc['kv_per_token_bytes']:,} bytes")
        print(f"    KV heads per GPU: {alloc['kv_heads_per_gpu']}")
        print(f"    Max tokens: {alloc['max_tokens']:,}")

## 2. ModelRunner — Forward Pass Execution

```python
# sglang/srt/model_executor/model_runner.py (simplified)

class ModelRunner:
    def __init__(self, model_config, server_args, tp_rank, tp_size):
        # Load the model
        self.model = self._load_model(model_config, server_args)
        
        # Attention backend (FlashInfer by default)
        self.attn_backend = FlashInferAttnBackend(
            model_config, server_args
        )
        
        # CUDA graph runner (for decode acceleration)
        if not server_args.disable_cuda_graph:
            self.cuda_graph_runner = CudaGraphRunner(
                self.model, server_args
            )
        
        # Logits processor and sampler
        self.logits_processor = LogitsProcessor(model_config)
        self.sampler = Sampler()
    
    def forward(self, batch: ForwardBatch):
        """Execute a forward pass."""
        
        if batch.forward_mode == ForwardMode.EXTEND:
            return self.forward_extend(batch)
        else:  # ForwardMode.DECODE
            return self.forward_decode(batch)
    
    def forward_extend(self, batch):
        """Forward pass for extend (prefill) mode."""
        # 1. Prepare inputs
        input_ids = batch.input_ids       # [total_extend_tokens]
        positions = batch.positions         # [total_extend_tokens]
        
        # 2. Set up attention metadata
        self.attn_backend.init_forward_metadata(
            batch, ForwardMode.EXTEND
        )
        
        # 3. Run model forward
        logits = self.model.forward(
            input_ids=input_ids,
            positions=positions,
            forward_batch=batch,
        )
        
        # 4. Process logits (only last token per sequence)
        logits = self.logits_processor.forward(logits, batch)
        
        # 5. Sample next tokens
        next_tokens = self.sampler.forward(logits, batch.sampling_params)
        
        return next_tokens
    
    def forward_decode(self, batch):
        """Forward pass for decode mode."""
        # Use CUDA graph if available
        if self.cuda_graph_runner and batch.batch_size <= self.cuda_graph_max_bs:
            return self.cuda_graph_runner.replay(batch)
        
        # Otherwise, regular forward
        input_ids = batch.input_ids    # [batch_size] (1 token per req)
        positions = batch.positions     # [batch_size]
        
        self.attn_backend.init_forward_metadata(
            batch, ForwardMode.DECODE
        )
        
        logits = self.model.forward(
            input_ids=input_ids,
            positions=positions,
            forward_batch=batch,
        )
        
        logits = self.logits_processor.forward(logits, batch)
        next_tokens = self.sampler.forward(logits, batch.sampling_params)
        
        return next_tokens
```

In [ ]:
# Simulate the forward pass pipeline

from enum import Enum

class ForwardMode(Enum):
    EXTEND = "extend"  # Prefill
    DECODE = "decode"  # Autoregressive

@dataclass
class ForwardBatch:
    """Simplified ForwardBatch."""
    forward_mode: ForwardMode
    input_ids: List[int]          # Token IDs to process
    positions: List[int]          # Position IDs
    batch_size: int               # Number of sequences
    extend_seq_lens: List[int] = None   # Lengths for extend
    seq_lens: List[int] = None          # Total sequence lengths


class ForwardPassSimulator:
    """Simulate the complete forward pass."""
    
    def __init__(self, model_name="Llama-3.1-8B", hidden_size=4096, vocab_size=128256):
        self.model_name = model_name
        self.hidden_size = hidden_size
        self.vocab_size = vocab_size
    
    def prepare_extend_batch(self, requests):
        """Prepare input for extend (prefill) mode."""
        all_input_ids = []
        all_positions = []
        extend_seq_lens = []
        
        for req in requests:
            prefix_len = req.get("prefix_len", 0)
            extend_ids = req["input_ids"][prefix_len:]  # Only non-cached tokens
            
            all_input_ids.extend(extend_ids)
            all_positions.extend(range(prefix_len, prefix_len + len(extend_ids)))
            extend_seq_lens.append(len(extend_ids))
        
        return ForwardBatch(
            forward_mode=ForwardMode.EXTEND,
            input_ids=all_input_ids,
            positions=all_positions,
            batch_size=len(requests),
            extend_seq_lens=extend_seq_lens,
        )
    
    def prepare_decode_batch(self, requests):
        """Prepare input for decode mode."""
        input_ids = [req["last_token_id"] for req in requests]
        positions = [req["current_pos"] for req in requests]
        seq_lens = [req["seq_len"] for req in requests]
        
        return ForwardBatch(
            forward_mode=ForwardMode.DECODE,
            input_ids=input_ids,
            positions=positions,
            batch_size=len(requests),
            seq_lens=seq_lens,
        )
    
    def display_batch(self, batch, label=""):
        """Display batch structure."""
        print(f"\n{label} Batch ({batch.forward_mode.value})")
        print(f"  Batch size: {batch.batch_size}")
        print(f"  Total tokens: {len(batch.input_ids)}")
        
        if batch.forward_mode == ForwardMode.EXTEND:
            print(f"  Per-sequence extend lengths: {batch.extend_seq_lens}")
            # Show ragged structure
            offset = 0
            for i, seq_len in enumerate(batch.extend_seq_lens):
                ids = batch.input_ids[offset:offset+seq_len]
                pos = batch.positions[offset:offset+seq_len]
                ids_str = str(ids[:5]) + ("..." if len(ids) > 5 else "")
                pos_str = str(pos[:5]) + ("..." if len(pos) > 5 else "")
                print(f"    Seq {i}: {seq_len} tokens, ids={ids_str}, pos={pos_str}")
                offset += seq_len
        else:
            print(f"  Input IDs: {batch.input_ids}")
            print(f"  Positions: {batch.positions}")
            print(f"  Seq lengths: {batch.seq_lens}")

# Demo
sim = ForwardPassSimulator()

# Extend batch
extend_requests = [
    {"input_ids": list(range(100)), "prefix_len": 50},   # 50 tokens cached, 50 to extend
    {"input_ids": list(range(30)), "prefix_len": 0},     # No cache, 30 to extend
    {"input_ids": list(range(80)), "prefix_len": 60},    # 60 cached, 20 to extend
]
extend_batch = sim.prepare_extend_batch(extend_requests)
sim.display_batch(extend_batch, "Extend")

# Decode batch
decode_requests = [
    {"last_token_id": 42, "current_pos": 100, "seq_len": 101},
    {"last_token_id": 55, "current_pos": 30, "seq_len": 31},
    {"last_token_id": 78, "current_pos": 80, "seq_len": 81},
    {"last_token_id": 99, "current_pos": 200, "seq_len": 201},
]
decode_batch = sim.prepare_decode_batch(decode_requests)
sim.display_batch(decode_batch, "Decode")

## 3. Input Preparation Details

### 3.1 Token IDs

For **extend**: concatenate all non-cached tokens into a flat tensor
```
Req A (cached=50, extend=30): [tok_50, tok_51, ..., tok_79]
Req B (cached=0, extend=60):  [tok_0, tok_1, ..., tok_59]

input_ids = [tok_50, ..., tok_79, tok_0, ..., tok_59]  # shape: [90]
```

For **decode**: one token per sequence
```
Req A (last token): [tok_100]
Req B (last token): [tok_75]

input_ids = [tok_100, tok_75]  # shape: [2]
```

### 3.2 Positions

Position IDs account for the full sequence length including cached prefix:
```
Req A (cached=50, extend=30): positions = [50, 51, ..., 79]
Req B (cached=0, extend=60):  positions = [0, 1, ..., 59]
```

### 3.3 Attention Metadata

FlashInfer needs to know:
- Where each sequence's KV-cache lives (page table)
- Sequence lengths for proper masking
- Which tokens are queries vs. keys

In [ ]:
# Visualize input preparation for both modes

def visualize_input_preparation():
    """Show how inputs are prepared for extend vs decode."""
    
    print("Input Preparation: Extend Mode")
    print("=" * 70)
    
    # Three requests with different cache hits
    extend_info = [
        {"rid": "A", "total": 100, "cached": 50, "extend": 50},
        {"rid": "B", "total": 60, "cached": 0, "extend": 60},
        {"rid": "C", "total": 80, "cached": 60, "extend": 20},
    ]
    
    total_extend = sum(r["extend"] for r in extend_info)
    
    print(f"\n  Batch: {len(extend_info)} sequences, {total_extend} total extend tokens")
    print()
    
    offset = 0
    for r in extend_info:
        cached_bar = "C" * min(r["cached"], 25)
        extend_bar = "E" * min(r["extend"], 25)
        
        if r["cached"] > 25:
            cached_bar = cached_bar[:22] + "..."
        if r["extend"] > 25:
            extend_bar = extend_bar[:22] + "..."
        
        print(f"  Req {r['rid']}: [{cached_bar}|{extend_bar}]")
        print(f"          Cached: {r['cached']} tokens (reuse KV), "
              f"Extend: {r['extend']} tokens (compute)")
        print(f"          Positions: [{r['cached']}, {r['cached']+1}, ..., {r['total']-1}]")
        print(f"          Offset in input_ids: [{offset}, {offset+r['extend']-1}]")
        offset += r["extend"]
    
    print(f"\n  Flattened input_ids shape: [{total_extend}]")
    print(f"  Flattened positions shape: [{total_extend}]")
    
    print("\n\nInput Preparation: Decode Mode")
    print("=" * 70)
    
    decode_info = [
        {"rid": "D", "seq_len": 150, "position": 150},
        {"rid": "E", "seq_len": 80, "position": 80},
        {"rid": "F", "seq_len": 200, "position": 200},
    ]
    
    print(f"\n  Batch: {len(decode_info)} sequences, 1 token each")
    print()
    
    for r in decode_info:
        context_bar = "K" * min(r["seq_len"] // 5, 20)
        print(f"  Req {r['rid']}: [{context_bar}...] + [Q]  (context={r['seq_len']}, query=1)")
        print(f"          Position: {r['position']}")
    
    print(f"\n  input_ids shape: [{len(decode_info)}]  (1 per sequence)")
    print(f"  positions shape: [{len(decode_info)}]")

visualize_input_preparation()

## 4. CUDA Graph Capture

CUDA graphs capture a sequence of GPU operations and **replay** them without CPU overhead.

### Why CUDA Graphs for Decode?

Decode is **latency-sensitive** (small workload per step), so CPU overhead matters:

```
Without CUDA graphs:
CPU: [launch kernel 1][launch kernel 2][launch kernel 3][launch kernel 4]
GPU:    [kernel 1]       [kernel 2]       [kernel 3]       [kernel 4]
     ↑ CPU launch overhead between each kernel

With CUDA graphs:
CPU: [replay graph]  (single launch!)
GPU: [kernel 1][kernel 2][kernel 3][kernel 4]  (no gaps)
```

### Capture Process

```python
# sglang/srt/model_executor/cuda_graph_runner.py (simplified)

class CudaGraphRunner:
    def __init__(self, model, max_batch_sizes):
        self.graphs = {}  # batch_size -> captured graph
        
        # Capture graphs for common batch sizes
        for bs in [1, 2, 4, 8, 16, 32, 64, 128]:
            if bs <= max(max_batch_sizes):
                self.graphs[bs] = self._capture(model, bs)
    
    def _capture(self, model, batch_size):
        # Create static input buffers
        static_input_ids = torch.zeros(batch_size, dtype=torch.long, device='cuda')
        static_positions = torch.zeros(batch_size, dtype=torch.long, device='cuda')
        
        # Warmup
        model.forward(static_input_ids, static_positions)
        
        # Capture
        graph = torch.cuda.CUDAGraph()
        with torch.cuda.graph(graph):
            static_output = model.forward(static_input_ids, static_positions)
        
        return {
            'graph': graph,
            'input_ids': static_input_ids,
            'positions': static_positions,
            'output': static_output,
        }
    
    def replay(self, batch):
        # Find the smallest captured graph that fits
        bs = self._round_up_bs(batch.batch_size)
        captured = self.graphs[bs]
        
        # Copy inputs into static buffers
        captured['input_ids'][:batch.batch_size].copy_(batch.input_ids)
        captured['positions'][:batch.batch_size].copy_(batch.positions)
        
        # Replay the graph (very fast!)
        captured['graph'].replay()
        
        # Read outputs
        return captured['output'][:batch.batch_size]
```

In [ ]:
# CUDA graph batch size selection

def round_up_batch_size(bs, captured_sizes=[1, 2, 4, 8, 16, 32, 64, 128]):
    """Round up to nearest captured batch size."""
    for size in captured_sizes:
        if bs <= size:
            return size
    return None  # Too large, can't use CUDA graph

print("CUDA Graph Batch Size Mapping")
print("=" * 50)
print(f"{'Actual BS':>12s} {'Captured BS':>12s} {'Padding':>10s}")
print("-" * 50)

for actual_bs in [1, 3, 5, 8, 12, 16, 24, 32, 50, 64, 100, 128, 200]:
    captured = round_up_batch_size(actual_bs)
    if captured:
        padding = captured - actual_bs
        pct = padding / captured * 100
        print(f"{actual_bs:12d} {captured:12d} {padding:6d} ({pct:.0f}%)")
    else:
        print(f"{actual_bs:12d} {'N/A (too large)':>12s}")

print("\nNote: Padding waste is minimal for power-of-2 batch sizes")
print("SGLang captures at: [1, 2, 4, 8, 16, 32, 64, 128, ...]")

## 5. Logits Processing and Sampling

After the model forward pass, we need to:
1. **Extract relevant logits** (last token per sequence for extend)
2. **Apply logits processors** (temperature, repetition penalty, etc.)
3. **Sample** the next token

```python
# sglang/srt/layers/logits_processor.py (simplified)

class LogitsProcessor:
    def forward(self, logits, batch):
        # For extend: take only the last token's logits per sequence
        if batch.forward_mode == ForwardMode.EXTEND:
            # logits shape: [total_extend_tokens, vocab_size]
            # We need: [batch_size, vocab_size]
            last_indices = torch.cumsum(
                torch.tensor(batch.extend_seq_lens), dim=0
            ) - 1
            logits = logits[last_indices]  # [batch_size, vocab_size]
        
        return logits

# sglang/srt/layers/sampler.py (simplified)

class Sampler:
    def forward(self, logits, sampling_params_list):
        results = []
        for i, params in enumerate(sampling_params_list):
            token_logits = logits[i]  # [vocab_size]
            
            # Apply temperature
            if params.temperature > 0:
                token_logits = token_logits / params.temperature
            
            # Apply top-p (nucleus sampling)
            if params.top_p < 1.0:
                token_logits = top_p_filter(token_logits, params.top_p)
            
            # Apply top-k
            if params.top_k > 0:
                token_logits = top_k_filter(token_logits, params.top_k)
            
            # Sample
            if params.temperature == 0:  # Greedy
                token_id = torch.argmax(token_logits).item()
            else:
                probs = torch.softmax(token_logits, dim=-1)
                token_id = torch.multinomial(probs, num_samples=1).item()
            
            results.append(token_id)
        
        return results
```

In [ ]:
# Demonstrate sampling strategies

import math
import random

def softmax(logits):
    max_val = max(logits)
    exp_vals = [math.exp(x - max_val) for x in logits]
    total = sum(exp_vals)
    return [v / total for v in exp_vals]

def apply_temperature(logits, temperature):
    if temperature == 0:
        return logits  # Greedy
    return [l / temperature for l in logits]

def top_p_filter(logits, top_p):
    probs = softmax(logits)
    sorted_indices = sorted(range(len(probs)), key=lambda i: probs[i], reverse=True)
    cumsum = 0
    keep = set()
    for idx in sorted_indices:
        cumsum += probs[idx]
        keep.add(idx)
        if cumsum >= top_p:
            break
    return [l if i in keep else -float('inf') for i, l in enumerate(logits)]

def sample(logits, temperature=1.0, top_p=1.0):
    logits = apply_temperature(logits, temperature)
    if top_p < 1.0:
        logits = top_p_filter(logits, top_p)
    
    if temperature == 0:
        return logits.index(max(logits))
    
    probs = softmax(logits)
    r = random.random()
    cumsum = 0
    for i, p in enumerate(probs):
        cumsum += p
        if cumsum >= r:
            return i
    return len(probs) - 1

# Demo with fake logits
vocab = ["the", "a", "cat", "dog", "sat", "on", "mat", "ran", "fast", "slow"]
raw_logits = [2.5, 1.0, 3.0, 2.8, 1.5, 0.5, 0.3, 1.8, 0.8, 0.2]

print("Sampling Strategy Comparison")
print("=" * 70)
print(f"Vocabulary: {vocab}")
print(f"Raw logits: {raw_logits}")
print(f"Raw probs:  {[f'{p:.3f}' for p in softmax(raw_logits)]}")

configs = [
    ("Greedy (temp=0)", 0, 1.0),
    ("Low temp (0.3)", 0.3, 1.0),
    ("Default (temp=1)", 1.0, 1.0),
    ("High temp (1.5)", 1.5, 1.0),
    ("Top-p=0.9", 1.0, 0.9),
    ("Top-p=0.5", 1.0, 0.5),
]

random.seed(42)
for name, temp, top_p in configs:
    samples = []
    for _ in range(100):
        idx = sample(list(raw_logits), temp, top_p)
        samples.append(vocab[idx])
    
    from collections import Counter
    counts = Counter(samples).most_common(5)
    dist = ", ".join(f"{w}:{c}%" for w, c in counts)
    print(f"\n  {name:20s}: {dist}")

## 6. Tensor Parallelism in SGLang

For large models, SGLang distributes the model across multiple GPUs:

```
Tensor Parallelism (TP=4):

GPU 0 (tp_rank=0):  GPU 1 (tp_rank=1):  GPU 2 (tp_rank=2):  GPU 3 (tp_rank=3):
┌──────────────┐   ┌──────────────┐   ┌──────────────┐   ┌──────────────┐
│ Embedding    │   │ (shared)     │   │ (shared)     │   │ (shared)     │
├──────────────┤   ├──────────────┤   ├──────────────┤   ├──────────────┤
│ Attn heads   │   │ Attn heads   │   │ Attn heads   │   │ Attn heads   │
│ 0-7          │   │ 8-15         │   │ 16-23        │   │ 24-31        │
├──────────────┤   ├──────────────┤   ├──────────────┤   ├──────────────┤
│ MLP cols     │   │ MLP cols     │   │ MLP cols     │   │ MLP cols     │
│ 0-3583       │   │ 3584-7167    │   │ 7168-10751   │   │ 10752-14335  │
├──────────────┤   ├──────────────┤   ├──────────────┤   ├──────────────┤
│ KV-cache     │   │ KV-cache     │   │ KV-cache     │   │ KV-cache     │
│ (heads 0-1)  │   │ (heads 2-3)  │   │ (heads 4-5)  │   │ (heads 6-7)  │
└──────┬───────┘   └──────┬───────┘   └──────┬───────┘   └──────┬───────┘
       │                  │                   │                  │
       └──────── All-Reduce (NCCL) ──────────┘                  │
                          └─────────── All-Reduce ──────────────┘
```

In [ ]:
# Demonstrate TP partitioning

def show_tp_partitioning(model_name, num_heads, kv_heads, hidden_size, mlp_size, tp_size):
    """Show how model layers are partitioned across TP ranks."""
    head_dim = hidden_size // num_heads
    heads_per_rank = num_heads // tp_size
    kv_heads_per_rank = kv_heads // tp_size
    mlp_per_rank = mlp_size // tp_size
    
    print(f"\n{model_name} with TP={tp_size}")
    print("=" * 60)
    print(f"  Total: {num_heads} Q heads, {kv_heads} KV heads, head_dim={head_dim}")
    print(f"  MLP intermediate: {mlp_size}")
    print()
    
    for rank in range(tp_size):
        q_start = rank * heads_per_rank
        q_end = (rank + 1) * heads_per_rank - 1
        kv_start = rank * kv_heads_per_rank
        kv_end = (rank + 1) * kv_heads_per_rank - 1
        mlp_start = rank * mlp_per_rank
        mlp_end = (rank + 1) * mlp_per_rank - 1
        
        q_params = heads_per_rank * head_dim * hidden_size * 2  # bytes
        kv_params = kv_heads_per_rank * head_dim * hidden_size * 2 * 2  # K+V
        mlp_params = mlp_per_rank * hidden_size * 2 * 3  # gate+up+down
        total_per_layer = (q_params + kv_params + mlp_params)
        
        print(f"  TP Rank {rank}:")
        print(f"    Q heads: [{q_start}-{q_end}] ({heads_per_rank} heads)")
        print(f"    KV heads: [{kv_start}-{kv_end}] ({kv_heads_per_rank} heads)")
        print(f"    MLP cols: [{mlp_start}-{mlp_end}]")
        print(f"    Params/layer: {total_per_layer/1e6:.1f} MB")

show_tp_partitioning("Llama-3.1-8B", 32, 8, 4096, 14336, 1)
show_tp_partitioning("Llama-3.1-8B", 32, 8, 4096, 14336, 2)
show_tp_partitioning("Llama-3.1-70B", 64, 8, 8192, 28672, 4)

## 7. Summary

### Key Takeaways

1. **TpModelWorker** manages model loading, memory allocation, and tensor parallel coordination
2. **ModelRunner** executes forward passes in two modes: extend (prefill) and decode
3. **Input preparation** differs significantly between extend and decode modes
4. **CUDA graphs** accelerate decode by eliminating CPU launch overhead
5. **Logits processing** extracts the right logits and applies sampling parameters
6. **Tensor parallelism** distributes attention heads and MLP across GPUs

### Next Chapter

Chapter 5.7 will dive into **FlashInfer Integration** — the attention kernel library that powers SGLang's performance.

## Exercises

### Exercise 1: Forward Pass Profiling
Design a profiling framework that measures:
- Time spent in each model layer
- Attention vs MLP compute ratio
- Memory bandwidth utilization

### Exercise 2: CUDA Graph Analysis
Measure the performance benefit of CUDA graphs for different batch sizes. At what batch size does the CPU overhead become negligible?

### Exercise 3: Custom Sampler
Implement a custom sampler that supports:
- Min-p sampling
- Mirostat sampling
- Beam search with n>1

In [ ]:
# Exercise 3 starter: Custom sampler

def min_p_sample(logits, min_p=0.05, temperature=1.0):
    """Min-p sampling: keep tokens with probability >= min_p * max_prob.
    
    TODO: Implement
    1. Apply temperature
    2. Convert to probabilities
    3. Find max probability
    4. Filter tokens with prob < min_p * max_prob
    5. Renormalize and sample
    """
    pass

print("Exercise 3: Implement min_p_sample()")